In [ ]:
import pandas as pd
from sqlalchemy import create_engine,VARCHAR,NUMERIC,INTEGER,DATE,DATETIME,String,text

conexaoDB = ('DRIVER={ODBC Driver 17 for SQL Server};''SERVER=DESKTOP-33OODCP;''DATABASE=AdventureWorksDW2019;''Trusted_Connection=yes;')
engine = create_engine(f'mssql+pyodbc:///?odbc_connect={conexaoDB}')

query = '''
SELECT 
ProductKey
,ProductAlternateKey
,FrenchProductName
,EnglishProductName
,Color 
,StandardCost
  FROM [AdventureWorksDW2019].[dbo].[DimProduct]
  where StandardCost is not null
'''

df = pd.read_sql_query(query, engine)

# Exiba  DataFrame
engine.dispose()
df.head()

In [ ]:
conexaoDB_destino = ('DRIVER={ODBC Driver 17 for SQL Server};''SERVER=DESKTOP-33OODCP;''DATABASE=Python;''Trusted_Connection=yes;')
engine_destino = create_engine(f'mssql+pyodbc:///?odbc_connect={conexaoDB_destino}')

#executar comando de delete
delete = text( """
delete  FROM [Python].[dbo].[produto_etl_historico] 
where color= 'Red'
"""
)
cursor = engine_destino.connect()
cursor.execute(delete)
cursor.commit()
cursor.close()

# Carga de dados 

tabela_destino= 'produto_etl_historico'
tipo_colnas = {        
    'ProductAlternateKey':VARCHAR(10),
    'Color':VARCHAR(15),
    'FrenchProductName':VARCHAR(50)
    }

df.to_sql(name= tabela_destino,con=engine_destino,if_exists='append' ,index=False , dtype=tipo_colnas)# enviar para banco 

engine_destino.dispose()

